In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install torch_geometric

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from sklearn.preprocessing import label_binarize
from sklearn.utils import resample
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from torch_geometric.nn import GATConv, GATv2Conv
from peft import LoraConfig, get_peft_model



In [ ]:
df=pd.read_csv('data.csv',index_col=0)

In [ ]:
import sys, os, re, csv, codecs, numpy as np, pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
nltk.download('stopwords')
import re
def removeUnnecessarySpaces(text):
    return re.sub(r'[\n\t\ ]+', ' ', text)

def removeNonArabicChar(text):
    return re.sub(r'[^0-9\u0600-\u06ff\u0750-\u077f\ufb50-\ufbc1\ufbd3-\ufd3f\ufd50-\ufd8f\ufd50-\ufd8f\ufe70-\ufefc\uFDF0-\uFDFD.0-9]+', ' ', text)

def sentTokenize(text):
    return text.replace(".", ". \n- ")

def clean(text):
    text = removeUnnecessarySpaces(text)
    text = removeNonArabicChar(text)
    text = removeUnnecessarySpaces(text)
    return sentTokenize(text)
##############################################################################
def clean_question(doc: str):
    """
    :param doc:
    :return: normalized string
    """
    chars = '[٠١٢٣٤٥٦٧٨٩0123456789[؟|$|.|!_،,@!#%^&*();<>":``.//\',\']'
    doc=clean(doc)
    STOPWORDS = set(stopwords.words('arabic'))
    doc = ' '.join([word for word in doc.split() if word not in STOPWORDS]) # delete stopwords from text
    doc = re.sub("[إأآا]", "ا", doc)
    doc = re.sub("ى", "ي", doc)
    doc = re.sub("ؤ", "ء", doc)
    doc = re.sub("ئ", "ء", doc)
    doc = re.sub("ة", "ه", doc)
    doc = re.sub("گ", "ك", doc)
    doc = re.sub(r'[^\w]+', ' ', str(doc))
    doc = re.sub(r'[a-zA-Z]', r'', doc)
    doc = re.sub(chars, r'', str(doc))
    noise = re.compile(""" ّ    | # Tashdid
                             َ    | # Fatha
                             ً    | # Tanwin Fath
                             ُ    | # Damma
                             ٌ    | # Tanwin Damm
                             ِ    | # Kasra
                             ٍ    | # Tanwin Kasr
                             ْ    | # Sukun
                             ـ     # Tatwil/Kashida
                         """, re.VERBOSE)
    doc = re.sub(noise, '', doc)
    doc = re.sub(r'(.)\1+', r"\1\1", doc) # Remove longation
    doc = re.sub(r'\s+', r' ', doc, flags=re.I)  # remove multiple spaces with single space
    
    doc = " ".join([word for word in doc.split() if len(word) > 2])
    #doc = doc.replace('\n', ' ')
    return doc
df['question']=[clean_question(str(x)) for x in tqdm(df.question)]

In [ ]:
df.rename(columns={'question': 'text', 'Cat1': 'main_label', 'Cat2': 'sub_label'}, inplace=True)
df

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)
train_df.head()

In [ ]:
main_label_encoder = LabelEncoder()
sub_label_encoder = LabelEncoder()

train_df['main_label'] = main_label_encoder.fit_transform(train_df['main_label'])
test_df['main_label'] = main_label_encoder.transform(test_df['main_label'])
val_df['main_label'] = main_label_encoder.transform(val_df['main_label'])

train_df['sub_label'] = sub_label_encoder.fit_transform(train_df['sub_label'])
test_df['sub_label'] = sub_label_encoder.transform(test_df['sub_label'])
val_df['sub_label'] = sub_label_encoder.transform(val_df['sub_label'])

num_main_labels = len(main_label_encoder.classes_)
num_sub_labels = len(sub_label_encoder.classes_)

print("Main label categories:", main_label_encoder.classes_)
print("Sub label categories:", sub_label_encoder.classes_)

In [ ]:
import torch
import spacy
import pandas as pd
from torch_geometric.data import Data

In [ ]:
!python -m spacy download xx_ent_wiki_sm

In [ ]:
# 🔹 Load Arabic NLP Model
try:
    nlp = spacy.load("xx_ent_wiki_sm")  # Universal model with Arabic support
except OSError:
    print("Please run: python -m spacy download xx_ent_wiki_sm")
    raise

In [ ]:
edge_index = []
label_to_idx = {}

# ✅ Assign an index to each unique label
print("Processing labels...")
all_labels = list(main_label_encoder.classes_) + list(sub_label_encoder.classes_)
for idx, label in enumerate(tqdm(all_labels, desc="Creating label index")):
    label_to_idx[label] = idx

# ✅ Process each text to extract dependency relationships
print("\nProcessing text and extracting dependencies...")
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Processing documents"):
    main_label = main_label_encoder.inverse_transform([row['main_label']])[0]
    sub_label = sub_label_encoder.inverse_transform([row['sub_label']])[0]
    
    main_idx, sub_idx = label_to_idx[main_label], label_to_idx[sub_label]
    
    # 🔹 Add co-occurrence edges
    edge_index.append([main_idx, sub_idx])
    edge_index.append([sub_idx, main_idx])
    
    # 🔹 Preprocess and extract dependency relations from Arabic text
    text = row['text']
    doc = nlp(text)
    
    # Arabic-specific dependency relations
    arabic_deps = [
        "nsubj",  # subject
        "obj",    # object
        "amod",   # adjectival modifier
        "iobj",   # indirect object
        "nmod",   # nominal modifier
        "compound"  # compound words (important for Arabic)
    ]
    
    for token in doc:
        if token.dep_ in arabic_deps:
            head, child = token.head.text, token.text
            
            # Handle Arabic word variants
            head = normalize_unicode(head)
            child = normalize_unicode(child)
            
            # Ensure words exist in labels (or are related to labels)
            for label, idx in label_to_idx.items():
                # Normalize label for comparison with Arabic text
                norm_label = normalize_unicode(label)
                if norm_label in head or norm_label in child:
                    edge_index.append([idx, main_idx])
                    edge_index.append([idx, sub_idx])

print("\nCreating graph tensors...")
# ✅ Convert edge list to tensor
edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

# ✅ Create random node features (will be updated by GAT)
node_features = torch.randn(len(all_labels), 16)

# ✅ Construct the graph data object
graph_data = Data(x=node_features, edge_index=edge_index)
print("\nGraph Summary:")
print(f"Graph Nodes: {graph_data.x.shape[0]} | Graph Edges: {graph_data.edge_index.shape[1]}")

In [ ]:


class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, input_dim, num_heads=4):
        super(MultiHeadAttentionLayer, self).__init__()
        self.num_heads = num_heads
        self.head_dim = input_dim // num_heads
        assert input_dim % num_heads == 0, "input_dim must be divisible by num_heads"
        
        self.query = nn.Linear(input_dim, input_dim)
        self.key = nn.Linear(input_dim, input_dim)
        self.value = nn.Linear(input_dim, input_dim)
        self.out_proj = nn.Linear(input_dim, input_dim)
        
    def forward(self, x):
        batch_size = x.size(0)
        
        # Transform input
        query = self.query(x).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        key = self.key(x).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        value = self.value(x).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Calculate attention scores
        scores = torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attention_weights = F.softmax(scores, dim=-1)
        
        # Apply attention to values
        context = torch.matmul(attention_weights, value)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.head_dim)
        
        output = self.out_proj(context)
        return output.squeeze(1)

class ContextGating(nn.Module):
    """Context gating mechanism for better feature modulation"""
    def __init__(self, dim):
        super(ContextGating, self).__init__()
        self.gate = nn.Linear(dim, dim)
        
    def forward(self, x):
        gates = torch.sigmoid(self.gate(x))
        return x * gates

class LabelAttention(nn.Module):
    def __init__(self, num_main_labels, hidden_dim):
        super(LabelAttention, self).__init__()
        self.attn = nn.Sequential(
            nn.Linear(num_main_labels, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * 2)
        )
        self.gate = ContextGating(hidden_dim * 2)
        
    def forward(self, main_label_pred, hidden_states):
        attn_weights = self.attn(main_label_pred)  # (batch_size, hidden_dim * 2)
        modulated = hidden_states * torch.sigmoid(attn_weights)
        return self.gate(modulated)  # Apply additional gating for controlled information flow

class MultiGATBlock(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, num_heads=4, dropout=0.2):
        super(MultiGATBlock, self).__init__()
        # First GAT layer with multiple heads
        self.gat1 = GATConv(in_dim, hidden_dim, heads=num_heads, dropout=dropout)
        # Second GAT layer uses GATv2 which has more expressive power
        self.gat2 = GATv2Conv(hidden_dim * num_heads, hidden_dim, heads=2, dropout=dropout)
        # Final GAT layer
        self.gat3 = GATConv(hidden_dim * 2, out_dim, dropout=dropout)
        
        self.norm1 = nn.LayerNorm(hidden_dim * num_heads)
        self.norm2 = nn.LayerNorm(hidden_dim * 2)
        self.norm3 = nn.LayerNorm(out_dim)
        
    def forward(self, x, edge_index):
        # First GAT layer
        x = self.gat1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.norm1(x)
        
        # Second GAT layer
        x = self.gat2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.norm2(x)
        
        # Third GAT layer
        x = self.gat3(x, edge_index)
        x = self.norm3(x)
        
        return x

class HierarchicalTextClassifier(nn.Module):
    def __init__(self, num_main_labels, num_sub_labels, gat_in_dim=16, gat_hidden_dim=32, 
                 text_embed_dim=768, lstm_hidden_dim=128, lstm_layers=2, dropout=0.3):
        super().__init__()

        # Arabic BERT model
        self.bert = AutoModel.from_pretrained("CAMeL-Lab/bert-base-arabic-camelbert-da")
        self.tokenizer = AutoTokenizer.from_pretrained("CAMeL-Lab/bert-base-arabic-camelbert-da")
        
        # Text projection layer
        self.text_projection = nn.Sequential(
            nn.Linear(text_embed_dim, text_embed_dim),
            nn.LayerNorm(text_embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Enhanced GAT for hierarchical label structure
        self.multi_gat = MultiGATBlock(gat_in_dim, gat_hidden_dim, gat_hidden_dim * 2)
        
        # LSTM for sequence modeling
        self.lstm = nn.LSTM(input_size=gat_hidden_dim * 2 + text_embed_dim, 
                           hidden_size=lstm_hidden_dim, 
                           num_layers=lstm_layers, 
                           batch_first=True, 
                           bidirectional=True,
                           dropout=dropout if lstm_layers > 1 else 0)

        # Multi-head attention for better feature representation
        self.attention = MultiHeadAttentionLayer(lstm_hidden_dim * 2)
        
        # Main label prediction with additional layer
        self.fc_main = nn.Sequential(
            nn.Linear(lstm_hidden_dim * 2, lstm_hidden_dim),
            nn.LayerNorm(lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, num_main_labels)
        )

        # Enhanced label attention mechanism
        self.label_attention = LabelAttention(num_main_labels, lstm_hidden_dim)
        
        # Sub-label prediction with enhanced architecture
        self.fc_sub = nn.Sequential(
            nn.Linear(lstm_hidden_dim * 2, lstm_hidden_dim),
            nn.LayerNorm(lstm_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden_dim, num_sub_labels)
        )
        
        # Residual connection for better gradient flow
        self.residual_proj = nn.Linear(text_embed_dim, lstm_hidden_dim * 2)
        
        # Focal loss parameters (can be adjusted during training)
        self.gamma = nn.Parameter(torch.ones(1) * 2.0)
        self.alpha = nn.Parameter(torch.ones(1) * 0.25)

    def forward(self, text, edge_index, node_features):
        # Process graph structure with enhanced GAT
        gat_out = self.multi_gat(node_features, edge_index)
        
        # Process text with CamelBERT
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(next(self.bert.parameters()).device) for k, v in inputs.items()}
        
        bert_outputs = self.bert(**inputs)
        text_cls = bert_outputs.last_hidden_state[:, 0, :]  # [CLS] token
        
        # Apply text projection
        text_features = self.text_projection(text_cls)
        
        # Save for residual connection
        residual = self.residual_proj(text_features)
        
        # Expand GAT output to match batch size
        batch_size = text_features.shape[0]
        gat_out_pooled = gat_out.mean(dim=0)
        gat_out_expanded = gat_out_pooled.unsqueeze(0).expand(batch_size, -1)
        
        # Combine text and graph features
        combined = torch.cat([gat_out_expanded, text_features], dim=1)
        combined = combined.unsqueeze(1)  # Add sequence dimension for LSTM
        
        # Process through LSTM
        lstm_out, _ = self.lstm(combined)
        lstm_out = lstm_out[:, -1, :]  # Get last output
        
        # Apply multi-head attention
        attended_features = self.attention(lstm_out.unsqueeze(1))
        
        # Add residual connection
        enhanced_features = attended_features + residual
        
        # Predict main labels
        main_label_pred = self.fc_main(enhanced_features)
        
        # Apply label attention for second-level classification
        attended_features = self.label_attention(main_label_pred, enhanced_features)
        
        # Predict sub-labels with enhanced features
        sub_label_pred = self.fc_sub(attended_features)
        
        return main_label_pred, sub_label_pred
    
    def focal_loss(self, pred, target, gamma=None, alpha=None):
        """Focal loss for better handling of class imbalance"""
        if gamma is None:
            gamma = self.gamma
        if alpha is None:
            alpha = self.alpha
            
        probs = torch.sigmoid(pred)
        pt = target * probs + (1 - target) * (1 - probs)
        weight = (1 - pt) ** gamma
        weight = alpha * weight
        loss = F.binary_cross_entropy_with_logits(pred, target, weight=weight, reduction='mean')
        return loss

In [ ]:
class TextDataset(Dataset):
    def __init__(self, df):
        self.texts = df['text'].tolist()
        self.main_labels = df['main_label'].tolist()
        self.sub_labels = df['sub_label'].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.main_labels[idx], self.sub_labels[idx]

train_dataset = TextDataset(train_df)
test_dataset = TextDataset(test_df)
val_dataset = TextDataset(val_df)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HierarchicalTextClassifier(num_main_labels, num_sub_labels).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5)

In [ ]:
epochs = 100
for epoch in range(epochs):
    # ---- Training ----
    model.train()
    total_loss = 0
    for texts, main_labels, sub_labels in tqdm(train_loader):
        main_labels, sub_labels = torch.tensor(main_labels).to(device), torch.tensor(sub_labels).to(device)
        optimizer.zero_grad()
        main_pred, sub_pred = model(texts, edge_index.to(device), graph_data.x.to(device))
        loss_main = criterion(main_pred, main_labels)
        loss_sub = criterion(sub_pred, sub_labels)
        loss = loss_main + loss_sub
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Train Loss: {total_loss / len(train_loader):.4f}")

    # ---- Validation ----
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for texts, main_labels, sub_labels in tqdm(val_loader):
            main_labels, sub_labels = torch.tensor(main_labels).to(device), torch.tensor(sub_labels).to(device)
            main_pred, sub_pred = model(texts, edge_index.to(device), graph_data.x.to(device))
            loss_main = criterion(main_pred, main_labels)
            loss_sub = criterion(sub_pred, sub_labels)
            val_loss += (loss_main + loss_sub).item()
    print(f"Epoch {epoch+1}, Val Loss: {val_loss / len(val_loader):.4f}")

In [ ]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print("-" * 50)
    print(f"Total Model Parameters:     {total_params:,}")
    print(f"Trainable Model Parameters: {trainable_params:,}")
    print("-" * 50)

count_parameters(model)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix

model.eval()
all_main_labels, all_sub_labels = [], []
all_main_preds, all_sub_preds = [], []
all_main_probs, all_sub_probs = [], []  # Store raw probabilities

with torch.no_grad():
    for texts, main_labels, sub_labels in test_loader:
        main_labels, sub_labels = main_labels.clone().detach().to(device), sub_labels.clone().detach().to(device)

        main_logits, sub_logits = model(texts, edge_index.to(device), graph_data.x.to(device))

        # Convert logits to probabilities
        main_probs = torch.softmax(main_logits, dim=1)
        sub_probs = torch.softmax(sub_logits, dim=1)

        # Get predicted class indices
        _, main_predicted = torch.max(main_probs, 1)
        _, sub_predicted = torch.max(sub_probs, 1)

        # Store actual labels and predictions
        all_main_labels.extend(main_labels.cpu().numpy())
        all_sub_labels.extend(sub_labels.cpu().numpy())
        all_main_preds.extend(main_predicted.cpu().numpy())
        all_sub_preds.extend(sub_predicted.cpu().numpy())
        all_main_probs.extend(main_probs.cpu().numpy())
        all_sub_probs.extend(sub_probs.cpu().numpy())

In [ ]:
def evaluate_metrics(true_labels, predicted_labels, predicted_probs, num_classes):
    precision = precision_score(true_labels, predicted_labels, average='macro', zero_division=0)
    recall = recall_score(true_labels, predicted_labels, average='macro')
    f1 = f1_score(true_labels, predicted_labels, average='macro')
    mcc = matthews_corrcoef(true_labels, predicted_labels)

    roc_auc = float('nan')
    if num_classes > 1:
        try:
            roc_auc = roc_auc_score(true_labels, predicted_probs, multi_class='ovo')
        except ValueError:
            pass

    return precision, recall, f1, mcc, roc_auc


# Convert lists to numpy arrays
all_main_labels = np.array(all_main_labels)
all_sub_labels = np.array(all_sub_labels)
all_main_preds = np.array(all_main_preds)
all_sub_preds = np.array(all_sub_preds)
all_main_probs = np.array(all_main_probs)
all_sub_probs = np.array(all_sub_probs)

# Get label names
main_label_names = list(main_label_encoder.classes_)
sub_label_names = list(sub_label_encoder.classes_)

# Compute metrics
print("Computing metrics...")
main_precision, main_recall, main_f1, main_mcc, main_roc_auc = evaluate_metrics(
    all_main_labels, all_main_preds, all_main_probs, num_main_labels)

sub_precision, sub_recall, sub_f1, sub_mcc, sub_roc_auc = evaluate_metrics(
    all_sub_labels, all_sub_preds, all_sub_probs, num_sub_labels)

In [ ]:
def build_hierarchy(main_label_encoder, sub_label_encoder, train_df):
    """
    Build parent → children mapping from training data.
    Returns:
        parent_of[sub_idx] = main_idx
    """
    parent_of = {}
    for _, row in train_df.iterrows():
        parent_of[int(row['sub_label'])] = int(row['main_label'])
    return parent_of


def hierarchical_precision_recall_f1(true_main, true_sub,
                                      pred_main, pred_sub,
                                      parent_of):
    """
    Computes hierarchical precision (hP), recall (hR), and F1 (hF1).

    For each sample, the 'true set' = {true_sub, true_main}
                    the 'pred set' = {pred_sub, pred_main}
    hP = |pred ∩ true| / |pred|
    hR = |pred ∩ true| / |true|
    hF1 = 2*hP*hR / (hP + hR)
    """
    hp_list, hr_list = [], []

    for tm, ts, pm, ps in zip(true_main, true_sub, pred_main, pred_sub):
        true_set = {('main', int(tm)), ('sub', int(ts))}
        pred_set = {('main', int(pm)), ('sub', int(ps))}

        # Propagate predicted sub-label up to its parent
        if int(ps) in parent_of:
            pred_set.add(('main', parent_of[int(ps)]))
        # Propagate true sub-label up to its parent
        if int(ts) in parent_of:
            true_set.add(('main', parent_of[int(ts)]))

        intersection = len(true_set & pred_set)
        hp_list.append(intersection / len(pred_set) if pred_set else 0.0)
        hr_list.append(intersection / len(true_set) if true_set else 0.0)

    hP = np.mean(hp_list)
    hR = np.mean(hr_list)
    hF1 = (2 * hP * hR / (hP + hR)) if (hP + hR) > 0 else 0.0
    return hP, hR, hF1


def lca_f1(true_main, true_sub, pred_main, pred_sub, parent_of):
    """
    LCA-F1: F1 computed at the Lowest Common Ancestor level.
    If pred_sub parent == true_sub parent → LCA is main level.
    If pred_sub == true_sub → LCA is sub level.
    Otherwise LCA is root (no credit).
    """
    scores = []
    for tm, ts, pm, ps in zip(true_main, true_sub, pred_main, pred_sub):
        ts, ps, tm, pm = int(ts), int(ps), int(tm), int(pm)
        if ts == ps:
            scores.append(1.0)          # exact sub-label match
        elif parent_of.get(ts) == parent_of.get(ps) and parent_of.get(ts) is not None:
            scores.append(0.5)          # same parent (main label match)
        else:
            scores.append(0.0)
    return np.mean(scores)

In [ ]:
# ─────────────────────────────────────────────
# 3. Calibration Metrics
# ─────────────────────────────────────────────

def expected_calibration_error(true_labels, predicted_probs, n_bins=15):
    """
    Expected Calibration Error (ECE).
    true_labels : 1-D array of integer class indices
    predicted_probs : 2-D array (n_samples, n_classes)
    """
    true_labels = np.array(true_labels)
    predicted_probs = np.array(predicted_probs)

    confidences = predicted_probs.max(axis=1)
    predictions = predicted_probs.argmax(axis=1)
    correct     = (predictions == true_labels).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc  = correct[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (mask.sum() / len(true_labels)) * abs(bin_acc - bin_conf)
    return ece


def multiclass_brier_score(true_labels, predicted_probs, num_classes):
    """
    Multiclass Brier Score (mean over classes).
    """
    true_labels     = np.array(true_labels)
    predicted_probs = np.array(predicted_probs)
    true_onehot     = label_binarize(true_labels, classes=np.arange(num_classes))

    # Handle binary case where label_binarize returns single column
    if true_onehot.shape[1] == 1:
        true_onehot = np.hstack([1 - true_onehot, true_onehot])

    brier = np.mean(np.sum((predicted_probs - true_onehot) ** 2, axis=1))
    return brier

In [ ]:
# ─────────────────────────────────────────────
# 4. Confidence Intervals via Stratified Bootstrap
# ─────────────────────────────────────────────

def bootstrap_ci(true_labels, predicted_labels, predicted_probs,
                 num_classes, n_bootstrap=1000, ci=95,
                 true_main=None, true_sub=None,
                 pred_main=None, pred_sub=None, parent_of=None):
    """
    Returns a dict of {metric_name: (mean, lower_ci, upper_ci)}.
    Uses stratified resampling (resample with stratify).
    """
    true_labels      = np.array(true_labels)
    predicted_labels = np.array(predicted_labels)
    predicted_probs  = np.array(predicted_probs)

    records = []
    alpha   = (100 - ci) / 2

    for _ in range(n_bootstrap):
        # Stratified bootstrap
        indices = resample(np.arange(len(true_labels)),
                           stratify=true_labels,
                           random_state=None)

        y_t  = true_labels[indices]
        y_p  = predicted_labels[indices]
        y_pr = predicted_probs[indices]

        acc  = np.mean(y_t == y_p)
        prec = precision_score(y_t, y_p, average='macro', zero_division=0)
        rec  = recall_score(y_t, y_p, average='macro', zero_division=0)
        f1   = f1_score(y_t, y_p, average='macro', zero_division=0)
        mcc  = matthews_corrcoef(y_t, y_p)
        ece  = expected_calibration_error(y_t, y_pr)
        brier = multiclass_brier_score(y_t, y_pr, num_classes)

        try:
            auc = roc_auc_score(y_t, y_pr, multi_class='ovo')
        except Exception:
            auc = float('nan')

        row = dict(acc=acc, prec=prec, rec=rec, f1=f1,
                   mcc=mcc, ece=ece, brier=brier, auc=auc)

        # Hierarchical metrics (if data provided)
        if parent_of is not None:
            tm_b  = np.array(true_main)[indices]
            ts_b  = np.array(true_sub)[indices]
            pm_b  = np.array(pred_main)[indices]
            ps_b  = np.array(pred_sub)[indices]
            hP, hR, hF1 = hierarchical_precision_recall_f1(
                tm_b, ts_b, pm_b, ps_b, parent_of)
            lca   = lca_f1(tm_b, ts_b, pm_b, ps_b, parent_of)
            row.update(hP=hP, hR=hR, hF1=hF1, lca_f1=lca)

        records.append(row)

    results = {}
    for key in records[0].keys():
        vals = np.array([r[key] for r in records if not np.isnan(r[key])])
        results[key] = (vals.mean(),
                        np.percentile(vals, alpha),
                        np.percentile(vals, 100 - alpha))
    return results

In [ ]:
print("\n Standard Mertic")
print("_" * 50)
print("_" * 50)
print("\nEvaluation Metrics - First Level")
print("-" * 50)
print(f"Accuracy:   {sum(all_main_labels == all_main_preds) / len(all_main_labels):.4f}")
print(f"Precision:  {main_precision:.4f}")
print(f"Recall:     {main_recall:.4f}")
print(f"F1 Score:   {main_f1:.4f}")
print(f"MCC:        {main_mcc:.4f}")
print(f"AUC:        {main_roc_auc:.4f}")

print("\nEvaluation Metrics - Second Level")
print("-" * 50)
print(f"Accuracy:   {sum(all_sub_labels == all_sub_preds) / len(all_sub_labels):.4f}")
print(f"Precision:  {sub_precision:.4f}")
print(f"Recall:     {sub_recall:.4f}")
print(f"F1 Score:   {sub_f1:.4f}")
print(f"MCC:        {sub_mcc:.4f}")
print(f"AUC:        {sub_roc_auc:.4f}")

In [ ]:
# --- Build hierarchy for hierarchical metrics ---
print("\n Hierarchical metrics ")
print("_"*50)
parent_of = build_hierarchy(main_label_encoder, sub_label_encoder, train_df)
# --- Hierarchical Metrics ---
hP, hR, hF1 = hierarchical_precision_recall_f1(
    all_main_labels, all_sub_labels,
    all_main_preds,  all_sub_preds,
    parent_of)

lca = lca_f1(all_main_labels, all_sub_labels,
             all_main_preds,  all_sub_preds, parent_of)

print(f"  Hier. Precision (hP) : {hP:.4f}")
print(f"  Hier. Recall    (hR) : {hR:.4f}")
print(f"  Hier. F1       (hF1) : {hF1:.4f}")
print(f"  LCA-F1               : {lca:.4f}")

In [ ]:
# --- Calibration Metrics ---
print("\n Calibration Metrics ")
print("_"*50)
print("First Level")
print("_"*35)
main_ece   = expected_calibration_error(all_main_labels, all_main_probs)
main_brier = multiclass_brier_score(all_main_labels, all_main_probs, num_main_labels)

sub_ece    = expected_calibration_error(all_sub_labels, all_sub_probs)
sub_brier  = multiclass_brier_score(all_sub_labels, all_sub_probs, num_sub_labels)
print(f"  ECE ↓              : {main_ece:.4f}")
print(f"  Brier Score ↓      : {main_brier:.4f}")
print("_"*35)
print("Second Level")
print(f"  ECE ↓              : {sub_ece:.4f}")
print(f"  Brier Score ↓      : {sub_brier:.4f}")
print("_"*50)

In [ ]:
import numpy as np
import torch
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# Set font globally
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]

def plot_confusion_matrix(y_true, y_pred, labels, title):
    """
    Plot publication-ready confusion matrix
    """
    cm = confusion_matrix(y_true, y_pred)
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.2), max(6, len(labels) * 1.0)))

    # Use a clean, academic colormap
    cmap = sns.color_palette("light:steelblue", as_cmap=True)
    sns.heatmap(
        cm_percent,
        annot=False,
        cmap=cmap,
        linewidths=0.5,
        linecolor='white',
        cbar=True,
        ax=ax,
        vmin=0,
        vmax=100
    )

    # Add count + percentage annotations
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = 'white' if cm_percent[i, j] > 60 else 'black'
            ax.text(
                j + 0.5, i + 0.5,
                f"{cm[i, j]}\n({cm_percent[i, j]:.1f}%)",
                ha='center', va='center',
                fontsize=10, fontweight='bold', color=color
            )

    # Formatting
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel("Predicted Label", fontsize=12, labelpad=10)
    ax.set_ylabel("True Label", fontsize=12, labelpad=10)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=10)
    ax.set_yticklabels(labels, rotation=0, fontsize=10)

    # Colorbar label
    cbar = ax.collections[0].colorbar
    cbar.set_label('Percentage (%)', fontsize=10)

    plt.tight_layout()
    plt.savefig(f"{title.replace(' ', '_')}.pdf", dpi=300, bbox_inches='tight')
    plt.savefig(f"{title.replace(' ', '_')}.png", dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {title.replace(' ', '_')}.pdf / .png")

# Plot confusion matrices
plot_confusion_matrix(all_main_labels, all_main_preds, main_label_names, " ")
plot_confusion_matrix(all_sub_labels, all_sub_preds, sub_label_names, " ")

In [ ]:
def print_ci_table(ci_dict, level_name="", ci_pct=95):
    print(f"\n{'='*60}")
    print(f"  {level_name}  —  {ci_pct}% Confidence Intervals (Stratified Bootstrap)")
    print(f"{'='*60}")
    print(f"  {'Metric':<18} {'Mean':>8}  [{' CI Lower':>9} , {'CI Upper':>9}]")
    print(f"  {'-'*54}")
    label_map = {
        'acc':    'Accuracy',
        'prec':   'Precision',
        'rec':    'Recall',
        'f1':     'F1-score',
        'mcc':    'MCC',
        'auc':    'AUC-ROC',
        'ece':    'ECE ↓',
        'brier':  'Brier Score ↓',
        'hP':     'Hier. Precision',
        'hR':     'Hier. Recall',
        'hF1':    'Hier. F1',
        'lca_f1': 'LCA-F1',
    }
    for k, (mean, lo, hi) in ci_dict.items():
        name = label_map.get(k, k)
        print(f"  {name:<18} {mean:>8.4f}  [{lo:>9.4f} , {hi:>9.4f}]")
    print(f"{'='*60}\n")


# ─────────────────────────────────────────────
# 5. Run Evaluation
# ─────────────────────────────────────────────

# Convert to numpy arrays
all_main_labels = np.array(all_main_labels)
all_sub_labels  = np.array(all_sub_labels)
all_main_preds  = np.array(all_main_preds)
all_sub_preds   = np.array(all_sub_preds)
all_main_probs  = np.array(all_main_probs)
all_sub_probs   = np.array(all_sub_probs)

main_label_names = list(main_label_encoder.classes_)
sub_label_names  = list(sub_label_encoder.classes_)

In [ ]:
# --- Bootstrap Confidence Intervals ---
print("\n[INFO] Running stratified bootstrap (n=1000) for Level 1...")
main_ci = bootstrap_ci(
    all_main_labels, all_main_preds, all_main_probs,
    num_main_labels, n_bootstrap=1000, ci=95,
    true_main=all_main_labels, true_sub=all_sub_labels,
    pred_main=all_main_preds,  pred_sub=all_sub_preds,
    parent_of=parent_of)

print_ci_table(main_ci, "Level 1 — Main Labels", ci_pct=95)

print("[INFO] Running stratified bootstrap (n=1000) for Level 2...")
sub_ci = bootstrap_ci(
    all_sub_labels, all_sub_preds, all_sub_probs,
    num_sub_labels, n_bootstrap=1000, ci=95,
    true_main=all_main_labels, true_sub=all_sub_labels,
    pred_main=all_main_preds,  pred_sub=all_sub_preds,
    parent_of=parent_of)

print_ci_table(sub_ci, "Level 2 — Sub Labels", ci_pct=95)

In [ ]:
def measure_latency(model, dataloader, device):
    model.eval()
    latencies = []
    
    # 1. GPU Warmup (Crucial for accurate timing)
    print("Warming up GPU...")
    dummy_texts, dummy_main, dummy_sub = next(iter(dataloader))
    with torch.no_grad():
        for _ in range(3):
            _ = model(dummy_texts, edge_index.to(device), graph_data.x.to(device))
    
    # 2. Measurement
    print("Measuring Latency on Test Set...")
    starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    
    with torch.no_grad():
        for texts, _, _ in dataloader:
            starter.record()
            
            # Forward pass
            _ = model(texts, edge_index.to(device), graph_data.x.to(device))
            
            ender.record()
            torch.cuda.synchronize() # Wait for GPU to finish before stopping the clock
            
            curr_time_ms = starter.elapsed_time(ender)
            # Divide by batch size to get per-sample latency
            latencies.append(curr_time_ms / len(texts))
            
    avg_latency = np.mean(latencies)
    p95_latency = np.percentile(latencies, 95)
    
    print("-" * 50)
    print(f"Average Latency (per sample): {avg_latency:.2f} ms")
    print(f"95th Percentile Latency:      {p95_latency:.2f} ms")
    print("-" * 50)

measure_latency(model, test_loader, device)

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

def measure_flops(model, dataloader, device):
    # Grab a single batch
    sample_texts, _, _ = next(iter(dataloader))
    batch_size = len(sample_texts)
    
    print(f"Profiling FLOPs for a batch of {batch_size} samples...")
    
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
        with_flops=True # Enables hardware FLOP counting
    ) as prof:
        with record_function("model_inference"):
            with torch.no_grad():
                model(sample_texts, edge_index.to(device), graph_data.x.to(device))

    # Calculate total FLOPs from all operations recorded
    total_flops = sum([event.flops for event in prof.key_averages() if event.flops > 0])
    
    print("-" * 50)
    print(f"Total FLOPs (Batch Size {batch_size}): {total_flops:,}")
    print(f"FLOPs per sample:           {total_flops / batch_size:,}")
    print("-" * 50)
    
    # Optional: Print the top 5 most expensive operations
    print("\nTop 5 Operations by CUDA Time:")
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=5))

measure_flops(model, test_loader, device)